# 12 - Pipelines and Sinks (distributed)

Demonstrates `SinkPipeline` and `ParquetPipeline` running inside a distributed Dask session fully managed by **boti-dask**.

Key boti-dask APIs used:
- `apply_recommended_dask_config()` — tasks-shuffle + memory/timeout config
- `DataHelper.session(cluster_factory=LocalCluster, cluster_kwargs={...})` — cluster lifecycle ownership

In [1]:
import datetime as dt
import os
from pathlib import Path
from tempfile import TemporaryDirectory

from dask.distributed import LocalCluster
from sqlalchemy import Date, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_dask import apply_recommended_dask_config
from boti_data import DataHelper, ParquetPipeline, SinkPipeline, available_sinks

In [2]:
class Base(DeclarativeBase):
    pass


class SourceEvent(Base):
    __tablename__ = "source_events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))


def seed(engine) -> None:
    Base.metadata.create_all(engine)
    with Session(engine) as session:
        session.add_all(
            [
                SourceEvent(id=1, event_date=dt.date(2026, 4, 15), status="active"),
                SourceEvent(id=2, event_date=dt.date(2026, 4, 16), status="inactive"),
                SourceEvent(id=3, event_date=dt.date(2026, 4, 17), status="active"),
            ]
        )
        session.commit()

In [3]:
tmp = TemporaryDirectory()
root = Path(tmp.name)
db_path = root / "pipeline_distributed_demo.db"
sqlite_dsn = f"sqlite:///{db_path}"
worker_dsn_env_var = "BOTI_NOTEBOOK_SQLITE_DSN"
prev_worker_dsn = os.environ.get(worker_dsn_env_var)
os.environ[worker_dsn_env_var] = sqlite_dsn

engine = create_engine(sqlite_dsn)
try:
    seed(engine)
finally:
    engine.dispose()

helper = DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    worker_connection_env_var=worker_dsn_env_var,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="source_events",
)
print("available sinks:", available_sinks())

available sinks: ('csv', 'jsonl', 'parquet')


## Run all three sinks inside a single distributed session

`apply_recommended_dask_config()` wraps the cluster start so config is applied first.  
Cluster kwargs (`n_workers`, `threads_per_worker`, `processes`) go in `cluster_kwargs=`.  
`DataHelper.session` owns the full cluster + client lifecycle.

In [4]:
with apply_recommended_dask_config():
    with DataHelper.session(
        cluster_factory=LocalCluster,
        cluster_kwargs={"n_workers": 2, "threads_per_worker": 1, "processes": False},
        verify_connectivity=True,
    ) as client:
        print("workers:", len(client.scheduler_info()["workers"]))

        csv_pipeline = SinkPipeline(
            helper,
            "csv",
            sink_config={
                "storage_path": str(root / "events_csv"),
                "partition_on": ["partition_date"],
                "project_root": root,
            },
            date_field="event_date",
        )
        jsonl_pipeline = SinkPipeline(
            helper,
            "jsonl",
            sink_config={
                "storage_path": str(root / "events_jsonl"),
                "partition_on": ["partition_date"],
                "project_root": root,
            },
            date_field="event_date",
        )
        parquet_pipeline = ParquetPipeline(
            helper,
            {
                "backend": "parquet",
                "storage_path": str(root / "events_parquet"),
                "partition_on": ["partition_date"],
                "project_root": root,
            },
            date_field="event_date",
        )

        try:
            csv_result = csv_pipeline.write(filters={"status__exact": "active"})
            jsonl_result = jsonl_pipeline.write(filters={"status__exact": "active"})
            mat_result = parquet_pipeline.materialize(
                filters={"status__exact": "active"}, reload=True
            )
        finally:
            csv_pipeline.close()
            jsonl_pipeline.close()
            parquet_pipeline.close()

csv_result, jsonl_result, mat_result

workers: 2


(SinkWriteResult(path='/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmppdmxyho2/events_csv', files=('/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmppdmxyho2/events_csv/partition_date=2026-04-15/part-0.csv', '/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmppdmxyho2/events_csv/partition_date=2026-04-17/part-0.csv')),
 SinkWriteResult(path='/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmppdmxyho2/events_jsonl', files=('/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmppdmxyho2/events_jsonl/partition_date=2026-04-15/part-0.jsonl', '/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmppdmxyho2/events_jsonl/partition_date=2026-04-17/part-0.jsonl')),
 ParquetMaterializationResult(path='/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmppdmxyho2/events_parquet', frame=Dask DataFrame Structure:
                   id           event_date  status   partition_date
 npartitions=1                                                     


In [5]:
reloaded = mat_result.frame.compute().sort_values("id").reset_index(drop=True)
assert mat_result.reloaded is True
assert reloaded["id"].tolist() == [1, 3]
assert len(csv_result.files) >= 1
assert len(jsonl_result.files) >= 1
reloaded

,id,event_date,status,partition_date
0,1,2026-04-15 00:00:00+00:00,active,2026-04-15
1,3,2026-04-17 00:00:00+00:00,active,2026-04-17


In [6]:
if prev_worker_dsn is None:
    os.environ.pop(worker_dsn_env_var, None)
else:
    os.environ[worker_dsn_env_var] = prev_worker_dsn

tmp.cleanup()
print("Cleaned up.")

Cleaned up.
